In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pickle

feature_bow = pickle.load(open("/content/drive/My Drive/Colab Notebooks/twitter-pligub-sentiment-analysis/model/feature-bow.p", 'rb'))
model_nb = pickle.load(open('/content/drive/My Drive/Colab Notebooks/twitter-pligub-sentiment-analysis/model/model-nb.p', 'rb'))
model_nn = pickle.load(open('/content/drive/My Drive/Colab Notebooks/twitter-pligub-sentiment-analysis/model/model-nn.p', 'rb'))

/usr/local/lib/python3.10/dist-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 0.22.2.post1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator MultinomialNB from version 0.22.2.post1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator LabelBinarizer from version 0.22.2.post1 when using version 1.5.2. This might lead

## Predict sentiment per kalimat

In [ ]:
!pip install Sastrawi
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords

original_text =  '''
Masakan ini ngga enak, cuih.
'''

factory = StemmerFactory()
stemmer = factory.create_stemmer()
stop_words = set(stopwords.words('indonesian'))

def preprocess(text):
    text = text.lower()
    new_text = []
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        t = re.sub(r'[^a-zA-Z\s]', '', t)
        if t not in stop_words and t != '@user':
            # Melakukan stemming
            t = stemmer.stem(t)
            new_text.append(t)
        #new_text.append(t)
    return " ".join(new_text)

text = preprocess(original_text)

In [ ]:
text_feature = feature_bow.transform([text])

In [ ]:
print("Text: ", str(original_text))
print(text)
print("Sentiment NB: ", model_nb.predict(text_feature)[0])
print("Sentiment NN: ", model_nn.predict(text_feature)[0])

Text:  
Masakan ini ngga enak, cuih.

masakan ngga enak cuih
Sentiment NB:  neutral
Sentiment NN:  positive


# Predict Sentiment kumpulan kalimat

In [ ]:
import pandas as pd
data_df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/twitter-pligub-sentiment-analysis/dataset/twitter-data-pilgub-jakarta.csv', sep=',')
data_df.head()

,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
0,1845269226778656886,Sun Oct 13 01:04:01 +0000 2024,0,Mantap! Jakarta Fund dapat menjadi sumber dana...,1845269226778656886,NaN,NaN,in,NaN,0,0,0,https://x.com/SaguKeju8/status/184526922677865...,1424746853346660357,SaguKeju8
1,1845268285547479132,Sun Oct 13 01:00:17 +0000 2024,1,Memilih pemimpin yang baik jujur dan tidak ser...,1845268285547479132,https://pbs.twimg.com/ext_tw_video_thumb/18452...,NaN,in,NaN,0,1,1,https://x.com/_riverheaven/status/184526828554...,975389335775199232,_riverheaven
2,1845268059487178997,Sun Oct 13 00:59:23 +0000 2024,0,Sekarang bukan lagi Doel anak sekolahan ... Sa...,1845268059487178997,https://pbs.twimg.com/ext_tw_video_thumb/18452...,NaN,in,NaN,0,0,0,https://x.com/BebySoSweet/status/1845268059487...,1708495620376571904,BebySoSweet
3,1845265669207753008,Sun Oct 13 00:53:20 +0000 2024,0,Acara yg mayoritas dihadiri generasi muda ini ...,1845266538607280641,https://pbs.twimg.com/media/GZu0HhuaAAAqjFw.jpg,HanifahAndini96,in,Indonesia,0,0,0,https://x.com/HanifahAndini96/status/184526653...,307856074,HanifahAndini96
4,1845265669207753008,Sun Oct 13 00:49:53 +0000 2024,1,Moment Cagub Pramono Anung saat bincang2 santa...,1845265669207753008,https://pbs.twimg.com/media/GZuzU17aAAA71HR.jpg,NaN,in,Indonesia,0,1,0,https://x.com/HanifahAndini96/status/184526566...,307856074,HanifahAndini96


In [ ]:
data_df = data_df[['full_text','locations']].rename(columns={'full_text': 'tweet'})

In [ ]:
data_df.head()

,tweet
0,Mantap! Jakarta Fund dapat menjadi sumber dana...
1,Memilih pemimpin yang baik jujur dan tidak ser...
2,Sekarang bukan lagi Doel anak sekolahan ... Sa...
3,Acara yg mayoritas dihadiri generasi muda ini ...
4,Moment Cagub Pramono Anung saat bincang2 santa...


In [ ]:
def predict_sentiment(sent):
    # cleansing
    text = preprocess(str(sent))
    # feature extraction
    text_feature = feature_bow.transform([text])
    # predict
    return model_nb.predict(text_feature)[0]

In [ ]:
data_df['tweet'] = data_df.tweet.apply(preprocess)
data_df.head()

,tweet
0,mantap jakarta fund sumber dana proyekproyek c...
1,pilih pimpin jujur serakah harus moga pramdoel...
2,doel anak sekolah jakartamenyala pramono rano...
3,acara yg mayoritas hadir generasi muda mrk har...
4,moment cagub pramono anung bincang santai dgn ...


In [ ]:
data_df['predict_sentiment'] = data_df.tweet.apply(predict_sentiment)

In [ ]:
data_df.head()

,tweet,predict_sentiment
0,mantap jakarta fund sumber dana proyekproyek c...,positive
1,pilih pimpin jujur serakah harus moga pramdoel...,positive
2,doel anak sekolah jakartamenyala pramono rano...,neutral
3,acara yg mayoritas hadir generasi muda mrk har...,neutral
4,moment cagub pramono anung bincang santai dgn ...,neutral


In [ ]:
data_df.to_csv('/content/drive/My Drive/Colab Notebooks/twitter-pligub-sentiment-analysis/dataset/dataset_predicted_sentiment.csv')